In [74]:
import csv
file_path = "loan_records.csv"

In [75]:
# 1. Tabular / CSV data processing

In [76]:
# Output CSV file for LoanAccount entity
output_file = "Result/LoanAccount.csv"

# Field indices based on Fannie Mae pipe-delimited format
DQ_INDEX = 39          # Delinquency Status
ZB_CODE_INDEX = 43     # Zero Balance Code


def extract_fields(line: str):
    """
    Parse a single pipe-delimited record line and extract
    LoanAccount related fields.
    """
    parts = line.rstrip("\n").split("|")

    # Clean the suffix to avoid quotes in accountID
    suffix = parts[0].strip().strip('"').strip("'")

    # Core identifiers
    loan_id = parts[1]                    # Loan Identifier
    account_id = f"{loan_id}_{suffix}"    # Reference Pool based Account ID

    # Financial attributes
    loan_balance_amount = parts[11]       # Current Actual UPB
    loan_age_months = parts[15]           # Loan Age in months
    remaining_term = parts[17]            # Remaining Months to Maturity
    current_rate = parts[8]               # Current Interest Rate

    # Raw status fields (not derived or normalized here)
    account_status = parts[DQ_INDEX]      # Delinquency Status
    has_status = parts[ZB_CODE_INDEX]     # Zero Balance Code

    return {
        "loanID": loan_id,
        "accountID": account_id,
        "loanBalanceAmount": loan_balance_amount,
        # "loanAgeMonths": loan_age_months,
        # "remainingTerm": remaining_term,
        # "currentRate": current_rate,
        "accountStatus": "Active",
        "hasStatus": "Active",
    }


# Read the input file
records = []
with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

# Skip header line and start processing from the second line
for line in lines[1:]:
    line = line.strip()
    if not line:
        continue

    # Each line format:
    # index <TAB> pipe-delimited-record
    if "\t" in line:
        _, raw_record = line.split("\t", 1)
    else:
        # If no tab exists, treat the whole line as the record
        raw_record = line

    # Extract structured fields from the raw record
    extracted = extract_fields(raw_record)
    records.append(extracted)


# Write extracted records to output CSV
with open(output_file, "w", newline="", encoding="utf-8") as out_f:
    writer = csv.DictWriter(
        out_f,
        fieldnames=[
            "loanID",
            "accountID",
            "loanBalanceAmount",
            # "loanAgeMonths",
            # "remainingTerm",
            # "currentRate",
            "accountStatus",
            "hasStatus",
        ],
    )
    writer.writeheader()
    writer.writerows(records)

print("Output written to:", output_file)

Output written to: Result/LoanAccount.csv


In [77]:
output_file = "Result/Payments.csv"

# ------------------------------------------
# Record level classification rules
# ------------------------------------------

BAD_ZB_CODES = {
    "03", "06", "09",
    "12", "13", "15", "16", "17",
    "96", "97", "98", "99",
}

DQ_INDEX = 39        # Delinquency status field index
ZB_CODE_INDEX = 43   # Zero balance code field index


def normalize_dq(raw_dq: str) -> str:
    """
    Normalize delinquency status.

    Rules:
    - Treat R, empty string, NA, or 0 as current status "00"
    - Otherwise keep the original value
    """
    dq = raw_dq.strip()
    if dq in {"R", "", "NA"}:
        return "00"
    if dq == "0":
        return "00"
    return dq


def classify_row_by_text_line(line: str):
    """
    Input:
        One full Fannie Mae pipe delimited text line

    Output:
        (label, dq_raw, dq_norm, zb_code)
    """
    parts = line.rstrip("\n").split("|")

    # Not enough fields to parse required indices
    if len(parts) <= max(DQ_INDEX, ZB_CODE_INDEX):
        return ("Unknown", None, None, None)

    dq_raw = parts[DQ_INDEX].strip()
    zb_code = parts[ZB_CODE_INDEX].strip()
    dq_norm = normalize_dq(dq_raw)

    # Case 1: Bad loan if zero balance code indicates default or termination
    if zb_code in BAD_ZB_CODES:
        return ("Bad", dq_raw, dq_norm, zb_code)

    try:
        dq_int = int(dq_norm)
    except ValueError:
        dq_int = 99

    # Case 2: Bad loan if delinquency is 3 months or more
    if dq_int >= 3:
        return ("missing", dq_raw, dq_norm, zb_code)

    # Case 3: Loan is due for 1 to 2 months delinquent
    if dq_norm in {"01", "02"}:
        return ("due", dq_raw, dq_norm, zb_code)

    # Case 4: Otherwise treat as due
    return ("due", dq_raw, dq_norm, zb_code)


# ------------------------------------------
# Loan level aggregation
# ------------------------------------------

def classify_loan_from_lines(lines):
    """
    Input:
        A list of raw text lines belonging to the same loan

    Output:
        loan_label, per_record_rows

    per_record_rows is a list of dicts.
    Each dict contains:
        period, label, dq_raw, dq_norm, zb, raw_line
    """
    has_bad = False
    has_avg = False
    rows = []

    for line in lines:
        line = line.strip()
        if not line:
            continue

        parts = line.split("|")
        period = parts[2] if len(parts) > 2 else ""

        label, dq_raw, dq_norm, zb = classify_row_by_text_line(line)

        rows.append(
            {
                "period": period,
                "label": label,
                "dq_raw": dq_raw,
                "dq_norm": dq_norm,
                "zb": zb,
                "raw_line": line,
            }
        )

        if label == "Bad":
            has_bad = True
        elif label == "Average":
            has_avg = True

    # Loan level aggregation rule
    if has_bad:
        loan_label = "Bad"
    elif has_avg:
        loan_label = "Average"
    else:
        loan_label = "Good"

    return loan_label, rows


def extract_fields(line: str):
    """
    Extract loan status category from a single record line.
    """
    parts = line.rstrip("\n").split("|")

    loan_id = parts[1]                 # Loan identifier
    account_id = parts[0]              # Reference pool identifier
    loan_balance_amount = parts[11]    # Current actual UPB
    loan_age_months = parts[15]        # Loan age in months
    remaining_term = parts[17]         # Remaining months to maturity
    current_rate = parts[8]            # Current interest rate

    label, dq_raw, dq_norm, zb_code = classify_row_by_text_line(line)
    account_status = label             # Good, due, or Bad

    return account_status


def compute_contract_scheduled_amount(parts):
    """
    Compute contractual scheduled monthly payment
    using original UPB, interest rate, and loan term.
    """
    try:
        original_upb = float(parts[9])     # Original UPB
        annual_rate = float(parts[7])      # Original interest rate
        term_months = int(parts[12])       # Original loan term
    except (ValueError, IndexError):
        return ""

    if term_months <= 0:
        return ""

    r = annual_rate / 100.0 / 12.0
    n = term_months

    if r == 0:
        m = original_upb / n
    else:
        try:
            m = original_upb * r * (1 + r) ** n / ((1 + r) ** n - 1)
        except ZeroDivisionError:
            return ""

    return f"{m:.2f}"


def extract_payment_fields(line: str):
    """
    Extract payment level fields from a single record line.
    """
    has_status = extract_fields(line)

    parts = line.rstrip("\n").split("|")

    scheduled_principal = parts[46] if len(parts) > 46 else ""
    unscheduled_principal = parts[48] if len(parts) > 48 else ""
    total_principal = parts[47] if len(parts) > 47 else ""

    # Payment history code placeholder
    payment_history_code = parts[1] + "hist2017"

    actual_amount = parts[11] if len(parts) > 11 else ""

    scheduled_amount_contract = compute_contract_scheduled_amount(parts)

    suffix = parts[0].strip().strip('"').strip("'")

    # Reporting period in MMYYYY format
    raw = parts[2].strip()

    payment_date = f"01/{raw[:2]}/{raw[2:]}"
    due_date = f"15/{raw[:2]}/{raw[2:]}"

    return {
        "paymentID": f"{parts[1]}_{parts[2]}",
        "accountID": f"{parts[1]}_{suffix}",
        "loanID": parts[1],
        "paymentDate": payment_date,
        "totalPrincipal": unscheduled_principal,
        "paymentHistoryCode": payment_history_code,
        "dueDate": due_date,
        "actualAmount": actual_amount,
        "scheduledAmount": scheduled_amount_contract,
        "transactionID": f"T{parts[1]}_{parts[2]}",
        "category": has_status,
        "stage": "MortgageServicingStage",
    }


# ------------------------------------------
# Read and process input file
# ------------------------------------------

records = []
with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

for line in lines[1:]:
    line = line.strip()
    if not line:
        continue

    # Each line format:
    # index <TAB> record
    if "\t" in line:
        _, raw_record = line.split("\t", 1)
    else:
        raw_record = line

    extracted = extract_payment_fields(raw_record)
    records.append(extracted)


# ------------------------------------------
# Write output CSV
# ------------------------------------------

with open(output_file, "w", newline="", encoding="utf-8") as out_f:
    writer = csv.DictWriter(
        out_f,
        fieldnames=[
            "paymentID",
            "accountID",
            "loanID",
            "paymentDate",
            "actualAmount",
            "paymentHistoryCode",
            "dueDate",
            "totalPrincipal",
            "scheduledAmount",
            "transactionID",
            "category",
            "stage",
        ]
    )
    writer.writeheader()
    writer.writerows(records)

print("Output written to:", output_file)

Output written to: Result/Payments.csv


In [78]:
output_file = "Result/Account.csv"

def extract_fields(line: str):
    """
    Parse a single pipe-delimited Fannie Mae record
    and extract Account-level fields.
    """
    parts = line.rstrip("\n").split("|")

    # Clean suffix to avoid quotes in accountID
    suffix = parts[0].strip().strip('"').strip("'")

    # Construct stable accountID
    account_id = f"{parts[1]}_{suffix}"

    # Current Interest Rate
    account_interest = parts[8] if len(parts) > 8 else ""

    return {
        "accountID": account_id,
        "accountInterest": account_interest,
        "balanceAmount": "",
        "personID": "",
    }


# ------------------------------------------
# Read file and deduplicate by accountID
# ------------------------------------------

records = []
seen_account_ids = set()

with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

# Skip header and start from second line
for line in lines[1:]:
    line = line.strip()
    if not line:
        continue

    # Each line may contain: index<TAB>record
    if "\t" in line:
        _, raw_record = line.split("\t", 1)
    else:
        raw_record = line

    extracted = extract_fields(raw_record)
    account_id = extracted["accountID"]

    # Keep only the first occurrence of each accountID
    if account_id in seen_account_ids:
        continue

    seen_account_ids.add(account_id)
    records.append(extracted)


# ------------------------------------------
# Write output CSV
# ------------------------------------------

with open(output_file, "w", newline="", encoding="utf-8") as out_f:
    writer = csv.DictWriter(
        out_f,
        fieldnames=[
            "accountID",
            "accountInterest",
            "balanceAmount",
            "personID",
        ],
    )
    writer.writeheader()
    writer.writerows(records)

print("Output written to:", output_file)

Output written to: Result/Account.csv


In [79]:
output_file = 'Result/Arrears.csv'

def parse_payment_history(ph: str):
    """
    Parse payment history string (index 40)
    Return limitCount and missedCount.
    """
    if not ph:
        return 0, 0

    missed = 0
    limit = 0

    for c in ph:
        if c.isdigit():
            v = int(c)
            if v > 0:
                missed += 1
            if v >= 3:
                limit += 1
        else:
            # X / NA / non-digit count as missed
            if c not in ("0",):
                missed += 1

    return limit, missed


def extract_fields(line: str):
    """
    Return limitCount, missedCount, applicationID, loanID
    """
    parts = line.rstrip("\n").split("|")

    # application_id = parts[0] if len(parts) > 0 else ""
    loan_id = parts[1] if len(parts) > 1 else ""

    payment_history = parts[40] if len(parts) > 40 else ""
    limitCount, missedCount = parse_payment_history(payment_history)

    return {
        "paymentDelta": limitCount,
        "missedCount": missedCount,
        "paymentID": f"{parts[1]}_{parts[2]}",
        "loanID": loan_id,
    }


records = []
with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

for line in lines[1:]:  # skip header
    line = line.strip()
    if not line:
        continue

    if "\t" in line:
        _, raw_record = line.split("\t", 1)
    else:
        raw_record = line

    extracted = extract_fields(raw_record)
    records.append(extracted)

with open(output_file, "w", newline="", encoding="utf-8") as out_f:
    writer = csv.DictWriter(
        out_f,
        fieldnames=["paymentDelta", "missedCount", "paymentID", "loanID"]
    )
    writer.writeheader()
    writer.writerows(records)

print("Output written to:", output_file)

Output written to: Result/Arrears.csv


In [80]:
output_file = 'Result/Address.csv'

def extract_fields(line: str):
    parts = line.rstrip("\n").split("|")

    suffix = parts[0].strip().strip('"').strip("'")
    accountID = f"{parts[1]}_{suffix}"

    postal_code = parts[32] if len(parts) > 32 else ""
    country = ""       # Fannie Mae dataset does not contain country
    street = ""                     # not provided in dataset
    city = ""                       # not provided in dataset
    orgID = ''
    personID = ''

    # application_id = parts[0] if len(parts) > 0 else ""
    # loan_id = parts[1] if len(parts) > 1 else ""

    return {
        "accountID": accountID,
        "postalCode": postal_code,
        "country": country,
        "street": street,
        "city": city,
        "orgID": orgID,
        "personID": personID,
    }


records = []
with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

for line in lines[1:]:  # skip header
    line = line.strip()
    if not line:
        continue

    if "\t" in line:
        _, raw_record = line.split("\t", 1)
    else:
        raw_record = line

    extracted = extract_fields(raw_record)
    records.append(extracted)

with open(output_file, "w", newline="", encoding="utf-8") as out_f:
    writer = csv.DictWriter(
        out_f,
        fieldnames=[
            "accountID",
            "postalCode",
            "country",
            "street",
            "city",
            "addressID",
            "orgID",
            "personID",
        ]
    )
    writer.writeheader()
    writer.writerows(records)

print("Output written to:", output_file)

Output written to: Result/Address.csv


In [81]:
output_file = "Result/Asset.csv"

def extract_fields(line: str):
    parts = line.rstrip("\n").split("|")

    # Listing based valuation proxy
    original_list_start = parts[63] if len(parts) > 63 else ""
    original_list_price = parts[64] if len(parts) > 64 else ""
    current_list_start = parts[65] if len(parts) > 65 else ""
    current_list_price = parts[66] if len(parts) > 66 else ""

    # Prefer current listing price and date if available
    market_value = current_list_price if current_list_price else original_list_price
    valuation_date = current_list_start if current_list_start else original_list_start

    # Identifiers
    suffix = parts[0].strip().strip('"').strip("'")
    loan_id = parts[1] if len(parts) > 1 else ""

    accountID = f"{loan_id}_{suffix}"
    assetID = f"{loan_id}asset"
    contractID = f"c{accountID}"
    addressID = ""

    return {
        "marketValue": market_value,
        "valuationDate": valuation_date,
        "marketValueAmount": market_value,
        "accountID": accountID,
        "assetID": assetID,
        "contractID": contractID,
        "addressID": addressID,
    }


# --------------------------------
# Read file and deduplicate by ID
# --------------------------------

records = []
seen_account_ids = set()

with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

# Skip header
for line in lines[1:]:
    line = line.strip()
    if not line:
        continue

    if "\t" in line:
        _, raw_record = line.split("\t", 1)
    else:
        raw_record = line

    extracted = extract_fields(raw_record)
    account_id = extracted["accountID"]

    # Keep only the first occurrence of each accountID
    if account_id in seen_account_ids:
        continue

    seen_account_ids.add(account_id)
    records.append(extracted)


# ---------------------------
# Write output CSV
# ---------------------------

with open(output_file, "w", newline="", encoding="utf-8") as out_f:
    writer = csv.DictWriter(
        out_f,
        fieldnames=[
            "marketValue",
            "valuationDate",
            "marketValueAmount",
            "accountID",
            "assetID",
            "contractID",
            "addressID",
        ],
    )
    writer.writeheader()
    writer.writerows(records)

print("Output written to:", output_file)

Output written to: Result/Asset.csv


In [82]:
output_file = "Result/Bank.csv"

def extract_fields(line: str):
    parts = line.rstrip("\n").split("|")

    # Clean suffix to avoid quotes
    suffix = parts[0].strip().strip('"').strip("'")

    loan_id = parts[1] if len(parts) > 1 else ""

    branch_id = f"{suffix}branch"
    org_id = f"FM{suffix}"

    return {
        "branchID": branch_id,
        "loanID": loan_id,
        "orgID": org_id,
    }


# --------------------------------
# Read file and deduplicate by loanID
# --------------------------------

records = []
seen_loan_ids = set()

with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

# Skip header
for line in lines[1:]:
    line = line.strip()
    if not line:
        continue

    # Handle index<TAB>record format
    if "\t" in line:
        _, raw_record = line.split("\t", 1)
    else:
        raw_record = line

    extracted = extract_fields(raw_record)
    loan_id = extracted["loanID"]

    # Keep only the first occurrence per loanID
    if loan_id in seen_loan_ids:
        continue

    seen_loan_ids.add(loan_id)
    records.append(extracted)


# ---------------------------
# Write output CSV
# ---------------------------

with open(output_file, "w", newline="", encoding="utf-8") as out_f:
    writer = csv.DictWriter(
        out_f,
        fieldnames=[
            "branchID",
            "loanID",
            "orgID",
        ],
    )
    writer.writeheader()
    writer.writerows(records)

print("Output written to:", output_file)

Output written to: Result/Bank.csv


In [83]:
output_file = "Result/CreditCheck.csv"

def extract_fields(line: str):
    parts = line.rstrip("\n").split("|")

    # Clean suffix
    suffix = parts[0].strip().strip('"').strip("'")

    loan_id = parts[1] if len(parts) > 1 else ""

    # Monthly application date in Fannie Mae format MMYYYY
    raw_month = parts[2].strip() if len(parts) > 2 else ""

    # Build IDs
    account_id = f"{loan_id}_{suffix}"
    event_id = f"CC{account_id}"
    application_id = f"a{loan_id}_{raw_month}"

    return {
        "hasStatus": "Pass",
        "accountID": account_id,
        "eventID": event_id,
        "applicationID": application_id,
    }


# --------------------------------
# Read file and deduplicate by accountID
# --------------------------------

records = []
seen_account_ids = set()

with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

# Skip header
for line in lines[1:]:
    line = line.strip()
    if not line:
        continue

    # Handle index<TAB>record format
    if "\t" in line:
        _, raw_record = line.split("\t", 1)
    else:
        raw_record = line

    extracted = extract_fields(raw_record)
    account_id = extracted["accountID"]

    # Keep only the first occurrence of each accountID
    if account_id in seen_account_ids:
        continue

    seen_account_ids.add(account_id)
    records.append(extracted)


# ---------------------------
# Write output CSV
# ---------------------------

with open(output_file, "w", newline="", encoding="utf-8") as out_f:
    writer = csv.DictWriter(
        out_f,
        fieldnames=[
            "hasStatus",
            "accountID",
            "eventID",
            "applicationID",
        ],
    )
    writer.writeheader()
    writer.writerows(records)

print("Output written to:", output_file)

Output written to: Result/CreditCheck.csv


In [84]:
output_file = "Result/Currency.csv"

# Static currency reference data
records = [
    {
        "currencyID": "cu001",
        "currencyName": "US Dollar",
        "currencySymbol": "$",
    },
    {
        "currencyID": "cu002",
        "currencyName": "Pound",
        "currencySymbol": "£",
    },
    {
        "currencyID": "cu003",
        "currencyName": "Euro",
        "currencySymbol": "€",
    },
]

# Write CSV
with open(output_file, "w", newline="", encoding="utf-8") as out_f:
    writer = csv.DictWriter(
        out_f,
        fieldnames=[
            "currencyID",
            "currencyName",
            "currencySymbol",
        ]
    )
    writer.writeheader()
    writer.writerows(records)

print("Output written to:", output_file)

Output written to: Result/Currency.csv


In [85]:
output_file = "Result/DocumentCheck.csv"

def extract_fields(line: str):
    parts = line.rstrip("\n").split("|")

    # Clean suffix
    suffix = parts[0].strip().strip('"').strip("'")

    loan_id = parts[1] if len(parts) > 1 else ""

    # Monthly application date MMYYYY
    raw_month = parts[2].strip() if len(parts) > 2 else ""

    # Build IDs
    account_id = f"{loan_id}_{suffix}"
    event_id = f"docCheckID{loan_id[-2:]}"
    application_id = f"a{loan_id}_{raw_month}"

    # Document ID placeholder rule
    document_id = parts[3].strip() if len(parts) > 3 else ""

    return {
        "hasStatus": "Pass",
        "accountID": account_id,
        "eventID": event_id,
        "applicationID": application_id,
        "documentID": '',
    }


# --------------------------------
# Read file and deduplicate by accountID
# --------------------------------

records = []
seen_account_ids = set()

with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

# Skip header
for line in lines[1:]:
    line = line.strip()
    if not line:
        continue

    # Handle index<TAB>record format
    if "\t" in line:
        _, raw_record = line.split("\t", 1)
    else:
        raw_record = line

    extracted = extract_fields(raw_record)
    account_id = extracted["accountID"]

    # Keep only the first occurrence per accountID
    if account_id in seen_account_ids:
        continue

    seen_account_ids.add(account_id)
    records.append(extracted)


# ---------------------------
# Write output CSV
# ---------------------------

with open(output_file, "w", newline="", encoding="utf-8") as out_f:
    writer = csv.DictWriter(
        out_f,
        fieldnames=[
            "hasStatus",
            "accountID",
            "eventID",
            "applicationID",
            "documentID",
        ],
    )
    writer.writeheader()
    writer.writerows(records)

print("Output written to:", output_file)

Output written to: Result/DocumentCheck.csv


In [86]:
output_file = "Result/Expense.csv"

records = [
    {
        "expenseID": "MG0_expense",
        "personID": "MG0",
        "expenseAmount": "14000",
    },
    {
        "expenseID": "MJ0_expense",
        "personID": "MJ0",
        "expenseAmount": "28000",
    },
    {
        "expenseID": "JE0_expense",
        "personID": "JE0",
        "expenseAmount": "21500",
    },
]

with open(output_file, "w", newline="", encoding="utf-8") as out_f:
    writer = csv.DictWriter(
        out_f,
        fieldnames=[
            "expenseID",
            "personID",
            "expenseAmount",
        ],
    )
    writer.writeheader()
    writer.writerows(records)

print("Output written to:", output_file)

Output written to: Result/Expense.csv


In [62]:
output_file = "Result/ForclosureEvent.csv"

records = [
    {
        "EventId": "EVT1A3FF2",
        "loanID": "45476383",
        "assetID": "45476383Asset",
        "contractID": "c45476383_1497",
        "requiredAmount": "192260",
    }
]

with open(output_file, "w", newline="", encoding="utf-8") as out_f:
    writer = csv.DictWriter(
        out_f,
        fieldnames=[
            "EventId",
            "loanID",
            "assetID",
            "contractID",
            "requiredAmount",
        ],
    )
    writer.writeheader()
    writer.writerows(records)

print("Output written to:", output_file)

Output written to: Result/ForclosureEvent.csv


In [63]:
output_file = "Result/Interest.csv"

def extract_status_interest(line: str):
    parts = line.rstrip("\n").split("|")

    interest_rate = parts[8] if len(parts) > 8 else ""
    loan_id = parts[1] if len(parts) > 1 else ""

    return {
        "hasStatus": "Fixed",
        "interestRate": interest_rate,
        "loanID": loan_id,
    }


# --------------------------------
# Read file and deduplicate by loanID
# --------------------------------

records = []
seen_loan_ids = set()

with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

# Skip header
for line in lines[1:]:
    line = line.strip()
    if not line:
        continue

    # Each line format: index<TAB>record
    if "\t" in line:
        _, raw_record = line.split("\t", 1)
    else:
        raw_record = line

    extracted = extract_status_interest(raw_record)
    loan_id = extracted["loanID"]

    # Keep only the first occurrence per loanID
    if loan_id in seen_loan_ids:
        continue

    seen_loan_ids.add(loan_id)
    records.append(extracted)


# ---------------------------
# Write output CSV
# ---------------------------

with open(output_file, "w", newline="", encoding="utf-8") as out_f:
    writer = csv.DictWriter(
        out_f,
        fieldnames=[
            "hasStatus",
            "interestRate",
            "loanID",
        ],
    )
    writer.writeheader()
    writer.writerows(records)

print("Output written to:", output_file)

Output written to: Result/Interest.csv


In [64]:
output_file = "Result/Liability.csv"

records = [
    {
        "liabilityAmount": "24000",
        "personID": "MG0",
        "liabilityID": "MG0_liability",
    },
    {
        "liabilityAmount": "88000",
        "personID": "MJ0",
        "liabilityID": "MJ0_liability",
    },
    {
        "liabilityAmount": "51500",
        "personID": "JE0",
        "liabilityID": "JE0_liability",
    },
]

with open(output_file, "w", newline="", encoding="utf-8") as out_f:
    writer = csv.DictWriter(
        out_f,
        fieldnames=[
            "liabilityAmount",
            "personID",
            "liabilityID",
        ],
    )
    writer.writeheader()
    writer.writerows(records)

print("Output written to:", output_file)

Output written to: Result/Liability.csv


In [65]:
import csv
import hashlib

output_file = "Result/LoanScheduale.csv"
# Field indices based on Fannie Mae pipe delimited format
POOL_INDEX = 0          # Reference Pool ID
LOAN_ID_INDEX = 1       # Loan Identifier
PERIOD_INDEX = 2        # Reporting Period, MMYYYY (for example 092017)
UPB_INDEX = 11          # Current Actual UPB


def extract_fields(line: str):
    """
    Parse a single pipe delimited record line and extract
    Payment schedule fields.
    """
    parts = line.rstrip("\n").split("|")

    if len(parts) <= max(POOL_INDEX, LOAN_ID_INDEX, PERIOD_INDEX, UPB_INDEX):
        return None

    suffix = parts[POOL_INDEX].strip().strip('"').strip("'")
    loan_id = parts[LOAN_ID_INDEX].strip()
    period = parts[PERIOD_INDEX].strip()           # MMYYYY
    remaining_credit = parts[UPB_INDEX].strip()    # UPB

    if not suffix or not loan_id or not period:
        return None

    account_id = f"{loan_id}_{suffix}"
    contract_id = f"c{account_id}"

    # Stable schedule id derived from accountID
    h = hashlib.md5(account_id.encode("utf-8")).hexdigest().upper()
    scheduale_id = f"SCH{h[:4]}"

    transaction_id = f"T{loan_id}_{period}"
    payment_id = f"{loan_id}_{period}"

    # Convert MMYYYY to 01 MM YYYY
    payment_date = ""
    if len(period) == 6 and period.isdigit():
        mm = period[:2]
        yyyy = period[2:]
        payment_date = f"01/{mm}/{yyyy}"

    return {
        "accountID": account_id,
        "schedualeID": scheduale_id,
        "contractID": contract_id,
        "transactionID": transaction_id,
        "remainingCredit": remaining_credit,
        "paymentDate": payment_date,
        "paymentID": payment_id,
        "period": period,  # helper for dedup, will not be written
    }


records = []
seen_keys = set()

with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

# Skip header line and start processing from the second line
for line in lines[1:]:
    line = line.strip()
    if not line:
        continue

    # Each line format:
    # index <TAB> pipe delimited record
    if "\t" in line:
        _, raw_record = line.split("\t", 1)
    else:
        raw_record = line

    extracted = extract_fields(raw_record)
    if not extracted:
        continue

    key = (extracted["accountID"], extracted["period"])
    if key in seen_keys:
        continue

    seen_keys.add(key)

    # Drop helper field before writing
    extracted.pop("period", None)
    records.append(extracted)

# Optional sort for readability
records.sort(key=lambda r: (r["accountID"], r["paymentDate"]))

with open(output_file, "w", newline="", encoding="utf-8") as out_f:
    writer = csv.DictWriter(
        out_f,
        fieldnames=[
            "accountID",
            "schedualeID",
            "contractID",
            "transactionID",
            "remainingCredit",
            "paymentDate",
            "paymentID",
        ],
    )
    writer.writeheader()
    writer.writerows(records)

print("Output written to:", output_file)

Output written to: Result/LoanScheduale.csv


In [66]:
output_file = "Result/MortgageApplication.csv"

records = [
    {
        "applicationID": "a45422752_122019",
        "loanID": "45422752",
        "contractID": "c45422752_1497",
        "coBorrowerCreditScore": "797",
        "deptToIncomeRatio": "28",
        "loanToValueReturn": "78",
        "combinedLoanToValueRation": "78",
        "loanPurpose": "R",
        "requestedAmount": "166000",
        "amountToPay": "236160",
        "applicationStatus": "P",
        "hasStatus": "0",
        "personID": "JE0",
        "stage": "ApplicationStage",
        "riskLevel": "LowLevelRisk",
        "mortgageID": "M1",
    },
    {
        "applicationID": "a45422765_012019",
        "loanID": "45422765",
        "contractID": "c45422765_1497",
        "coBorrowerCreditScore": "714",
        "deptToIncomeRatio": "41",
        "loanToValueReturn": "83",
        "combinedLoanToValueRation": "83",
        "loanPurpose": "R",
        "requestedAmount": "228000",
        "amountToPay": "307200",
        "applicationStatus": "P",
        "hasStatus": "0",
        "personID": "MJ0",
        "stage": "ApplicationStage",
        "riskLevel": "LowLevelRisk",
        "mortgageID": "M2",
    },
    {
        "applicationID": "a45476383_012019",
        "loanID": "45476383",
        "contractID": "c45476383_1497",
        "coBorrowerCreditScore": "700",
        "deptToIncomeRatio": "35",
        "loanToValueReturn": "80",
        "combinedLoanToValueRation": "80",
        "loanPurpose": "R",
        "requestedAmount": "148000",
        "amountToPay": "210480",
        "applicationStatus": "P",
        "hasStatus": "2",
        "personID": "MG0",
        "stage": "ApplicationStage",
        "riskLevel": "LowLevelRisk",
        "mortgageID": "M3",
    },
]

with open(output_file, "w", newline="", encoding="utf-8") as out_f:
    writer = csv.DictWriter(
        out_f,
        fieldnames=[
            "applicationID",
            "loanID",
            "contractID",
            "coBorrowerCreditScore",
            "deptToIncomeRatio",
            "loanToValueReturn",
            "combinedLoanToValueRation",
            "loanPurpose",
            "requestedAmount",
            "amountToPay",
            "applicationStatus",
            "hasStatus",
            "personID",
            "stage",
            "riskLevel",
            "mortgageID",
        ],
    )
    writer.writeheader()
    writer.writerows(records)

print("Output written to:", output_file)

Output written to: Result/MortgageApplication.csv


In [67]:
output_file = "Result/MortgageContract.csv"

records = [
    {
        "contractID": "c45422752_1497",
        "loanID": "45422752",
        "personID": "JE0",
        "orgID": "FM1497",
        "principalAmount": "236160",
        "mortgageInterestRate": "0.0375",
        "loanTermMonths": "240",
        "firstPaymentDate": "15/09/2017",
        "maturityDate": "15/09/2037",
        "amortizationType": "FRM",
        "prepayPenalty": "FALSE",
        "contractStatus": "accepted",
        "downpayment": "46820.51",
        "arrearsLimitCount": "3",
        "hasStatus": "0",
        "effectiveDate": "15/09/2017",
        "stage": "UnderwritingStage",
        "riskLevel": "LowRiskLevel",
        "currencyID": "cu001",
        "mortgageID": "M1",
        "guarontorID": "ML0",
    },
    {
        "contractID": "c45422765_1497",
        "loanID": "45422765",
        "personID": "MJ0",
        "orgID": "FM1497",
        "principalAmount": "307200",
        "mortgageInterestRate": "0.03125",
        "loanTermMonths": "180",
        "firstPaymentDate": "15/09/2017",
        "maturityDate": "15/09/2032",
        "amortizationType": "FRM",
        "prepayPenalty": "FALSE",
        "contractStatus": "accepted",
        "downpayment": "46698.8",
        "arrearsLimitCount": "2",
        "hasStatus": "0",
        "effectiveDate": "15/09/2017",
        "stage": "UnderwritingStage",
        "riskLevel": "LowRiskLevel",
        "currencyID": "cu002",
        "mortgageID": "M2",
        "guarontorID": "NJ0",
    },
    {
        "contractID": "c45476383_1497",
        "loanID": "45476383",
        "personID": "MG0",
        "orgID": "FM1497",
        "principalAmount": "148000",
        "mortgageInterestRate": "0.0375",
        "loanTermMonths": "240",
        "firstPaymentDate": "15/09/2017",
        "maturityDate": "15/09/2037",
        "amortizationType": "FRM",
        "prepayPenalty": "FALSE",
        "contractStatus": "accepted",
        "downpayment": "37000",
        "arrearsLimitCount": "2",
        "hasStatus": "2",
        "effectiveDate": "15/09/2017",
        "stage": "UnderwritingStage",
        "riskLevel": "LowRiskLevel",
        "currencyID": "cu003",
        "mortgageID": "M3",
        "guarontorID": "KA0",
    },
]

with open(output_file, "w", newline="", encoding="utf-8") as out_f:
    writer = csv.DictWriter(
        out_f,
        fieldnames=[
            "contractID",
            "loanID",
            "personID",
            "orgID",
            "principalAmount",
            "mortgageInterestRate",
            "loanTermMonths",
            "firstPaymentDate",
            "maturityDate",
            "amortizationType",
            "prepayPenalty",
            "contractStatus",
            "downpayment",
            "arrearsLimitCount",
            "hasStatus",
            "effectiveDate",
            "stage",
            "riskLevel",
            "currencyID",
            "mortgageID",
            "guarontorID",
        ],
    )
    writer.writeheader()
    writer.writerows(records)

print("Output written to:", output_file)

Output written to: Result/MortgageContract.csv


In [68]:
output_file = "Result/Organisation.csv"

records = [
    {
        "orgType": "R",
        "orgName": "Fannie Mae",
        "orgID": "FM1497",
        "addressID": "12564B1B",
        "branchID": "1497branch",
    }
]

with open(output_file, "w", newline="", encoding="utf-8") as out_f:
    writer = csv.DictWriter(
        out_f,
        fieldnames=[
            "orgType",
            "orgName",
            "orgID",
            "addressID",
            "branchID",
        ],
    )
    writer.writeheader()
    writer.writerows(records)

print("Output written to:", output_file)

Output written to: Result/Organisation.csv


In [69]:
output_file = "Result/Product.csv"

records = [
    {
        "orgID": "FM1497",
        "productID": "MG001",
        "productName": "Mortgage",
    },
    {
        "orgID": "FM1497",
        "productID": "FX001",
        "productName": "FX",
    },
    {
        "orgID": "SC175",
        "productID": "INC001",
        "productName": "Insurance",
    },
]

with open(output_file, "w", newline="", encoding="utf-8") as out_f:
    writer = csv.DictWriter(
        out_f,
        fieldnames=[
            "orgID",
            "productID",
            "productName",
        ],
    )
    writer.writeheader()
    writer.writerows(records)

print("Output written to:", output_file)

Output written to: Result/Product.csv


In [70]:
output_file = "Result/PropertyInspection.csv"

records = [
    {
        "eventID": "PI83asset1",
        "inspectionDate": "10/07/2017",
        "applicationID": "a45476383_032018",
        "assessmentID": "EVT1A3F",
    },
    {
        "eventID": "PI65asset1",
        "inspectionDate": "15/06/2017",
        "applicationID": "a45422765_022018",
        "assessmentID": "EVT4C7Q",
    },
    {
        "eventID": "PI52asset1",
        "inspectionDate": "01/08/2017",
        "applicationID": "a45422752_122020",
        "assessmentID": "EVT9M2K",
    },
]

with open(output_file, "w", newline="", encoding="utf-8") as out_f:
    writer = csv.DictWriter(
        out_f,
        fieldnames=[
            "eventID",
            "inspectionDate",
            "applicationID",
            "assessmentID",
        ],
    )
    writer.writeheader()
    writer.writerows(records)

print("Output written to:", output_file)

Output written to: Result/PropertyInspection.csv


In [71]:
output_file = "Result/RecordCheck.csv"

records = [
    {
        "hasStatus": "Active",
        "recordStatus": "Verified",
        "applicationID": "a45476383_012018",
        "EventID": "RC_EVT1A3F",
    },
    {
        "hasStatus": "Closed",
        "recordStatus": "Verified",
        "applicationID": "a45422765_012021",
        "EventID": "RC_EVT4C7Q",
    },
    {
        "hasStatus": "Pending",
        "recordStatus": "Verified",
        "applicationID": "a45422752_102020",
        "EventID": "RC_EVT9M2K",
    },
]

with open(output_file, "w", newline="", encoding="utf-8") as out_f:
    writer = csv.DictWriter(
        out_f,
        fieldnames=[
            "hasStatus",
            "recordStatus",
            "applicationID",
            "EventID",
        ],
    )
    writer.writeheader()
    writer.writerows(records)

print("Output written to:", output_file)

Output written to: Result/RecordCheck.csv


In [72]:
output_file = "Result/Transaction.csv"

# Field indices based on Fannie Mae pipe-delimited format
POOL_INDEX = 0
LOAN_ID_INDEX = 1
PERIOD_INDEX = 2            # Reporting period in MMYYYY format

# Indices for computing contractual monthly payment (P&I)
ORIGINAL_UPB_INDEX = 9      # Original UPB
ANNUAL_RATE_INDEX = 7       # Original Interest Rate (percentage)
TERM_MONTHS_INDEX = 12      # Original Loan Term (months)


def compute_required_amount(line: str):
    """
    Compute contractual monthly payment (principal + interest).
    """
    parts = line.rstrip("\n").split("|")

    try:
        original_upb = float(parts[ORIGINAL_UPB_INDEX])
        annual_rate = float(parts[ANNUAL_RATE_INDEX])
        term_months = int(parts[TERM_MONTHS_INDEX])
    except (ValueError, IndexError):
        return ""

    if term_months <= 0:
        return ""

    r = annual_rate / 100.0 / 12.0
    n = term_months

    if r == 0:
        m = original_upb / n
    else:
        try:
            m = original_upb * r * (1 + r) ** n / ((1 + r) ** n - 1)
        except ZeroDivisionError:
            return ""

    return f"{m:.2f}"


def extract_fields(line: str):
    """
    Parse a single pipe-delimited record line and extract
    transaction-level fields.
    """
    parts = line.rstrip("\n").split("|")

    if len(parts) <= max(POOL_INDEX, LOAN_ID_INDEX, PERIOD_INDEX):
        return None

    loan_id = parts[LOAN_ID_INDEX].strip()
    period = parts[PERIOD_INDEX].strip()    # MMYYYY

    if not loan_id or not period:
        return None

    # Compute monthly payment amount
    amount = compute_required_amount(line)
    if not amount:
        return None

    payment_id = f"{loan_id}_{period}"
    transaction_id = f"T{loan_id}_{period}"

    # Convert MMYYYY to DD/MM/YYYY (first day of month)
    transaction_date = ""
    if len(period) == 6 and period.isdigit():
        mm = period[:2]
        yyyy = period[2:]
        transaction_date = f"01/{mm}/{yyyy}"

    return {
        "transactionType": "loan_Payment",
        "hasAmount": amount,
        "paymentID": payment_id,
        "transactionID": transaction_id,
        "transactionDate": transaction_date,
        # Helper fields for deduplication
        "loanID": loan_id,
        "period": period,
    }


# ---------------------------
# Read input file
# ---------------------------

records = []
seen = set()

with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

# Skip header
for line in lines[1:]:
    line = line.strip()
    if not line:
        continue

    # Each line format: index<TAB>record OR record
    if "\t" in line:
        _, raw_record = line.split("\t", 1)
    else:
        raw_record = line

    extracted = extract_fields(raw_record)
    if not extracted:
        continue

    # Deduplicate by (loanID, period)
    key = (extracted["loanID"], extracted["period"])
    if key in seen:
        continue
    seen.add(key)

    extracted.pop("loanID", None)
    extracted.pop("period", None)

    records.append(extracted)

# Optional: sort by transaction date
records.sort(key=lambda r: r["transactionDate"])

# ---------------------------
# Write output CSV
# ---------------------------

with open(output_file, "w", newline="", encoding="utf-8") as out_f:
    writer = csv.DictWriter(
        out_f,
        fieldnames=[
            "transactionType",
            "hasAmount",
            "paymentID",
            "transactionID",
            "transactionDate",
        ],
    )
    writer.writeheader()
    writer.writerows(records)

print("Output written to:", output_file)

Output written to: Result/Transaction.csv


In [73]:
output_file = "Result/ValueAssessment.csv"

records = [
    {
        "assessedValueAmount": "325000",
        "EventID": "EVT1A3F",
        "assetID": "45476383Asset",
        "applicationID": "a45476383_042019",
        "reportID": "VAR83",
    },
    {
        "assessedValueAmount": "410000",
        "EventID": "EVT4C7Q",
        "assetID": "45422765Asset",
        "applicationID": "a45422765_052018",
        "reportID": "VAR65",
    },
    {
        "assessedValueAmount": "287500",
        "EventID": "EVT9M2K",
        "assetID": "45422752Asset",
        "applicationID": "a45422752_102017",
        "reportID": "VAR52",
    },
]

with open(output_file, "w", newline="", encoding="utf-8") as out_f:
    writer = csv.DictWriter(
        out_f,
        fieldnames=[
            "assessedValueAmount",
            "EventID",
            "assetID",
            "applicationID",
            "reportID",
        ],
    )
    writer.writeheader()
    writer.writerows(records)

print("Output written to:", output_file)

Output written to: Result/ValueAssessment.csv
